# Phoenemas definition

In [1]:
"""
Инвентарь фонем русского языка.
44 фонемы + специальные токены (blank для CTC, тишина).
"""

# Русские фонемы (упрощённая транскрипция)
RUSSIAN_PHONEMES = [
    # === ГЛАСНЫЕ (6 основных) ===
    "а",   # [a]  — мАк
    "э",   # [e]  — этот
    "и",   # [i]  — ИгЛа
    "о",   # [o]  — Он
    "у",   # [u]  — Ум
    "ы",   # [ɨ]  — бЫк

    # === СОГЛАСНЫЕ — ШУМНЫЕ СМЫЧНЫЕ ===
    "п",   # [p]  — Пас
    "б",   # [b]  — Бас
    "т",   # [t]  — Там
    "д",   # [d]  — Дом
    "к",   # [k]  — Кот
    "г",   # [ɡ]  — Год

    # === МЯГКИЕ ПАРЫ СМЫЧНЫХ ===
    "п'",  # [pʲ] — Пить
    "б'",  # [bʲ] — Бить
    "т'",  # [tʲ] — Тесто
    "д'",  # [dʲ] — День
    "к'",  # [kʲ] — Кит
    "г'",  # [ɡʲ] — Гиря

    # === ФРИКАТИВНЫЕ ===
    "ф",   # [f]  — Фон
    "в",   # [v]  — Вот
    "с",   # [s]  — Сон
    "з",   # [z]  — Зонт
    "ш",   # [ʂ]  — Шум
    "ж",   # [ʐ]  — Жук
    "х",   # [x]  — Ход
    "щ",   # [ɕː] — Щи

    # === МЯГКИЕ ПАРЫ ФРИКАТИВНЫХ ===
    "ф'",  # [fʲ] — Февраль
    "в'",  # [vʲ] — Вить
    "с'",  # [sʲ] — Синий
    "з'",  # [zʲ] — Зима
    "х'",  # [xʲ] — Химия

    # === АФФРИКАТЫ ===
    "ц",   # [ts] — Цапля
    "ч",   # [tɕ] — Час

    # === НОСОВЫЕ ===
    "м",   # [m]  — Мама
    "н",   # [n]  — Нос
    "м'",  # [mʲ] — Мир
    "н'",  # [nʲ] — Нить

    # === БОКОВЫЕ / ДРОЖАЩИЕ ===
    "л",   # [l]  — Лук
    "р",   # [r]  — Рот
    "л'",  # [lʲ] — Лить
    "р'",  # [rʲ] — Река

    # === АППРОКСИМАНТЫ ===
    "й",   # [j]  — Йод

    # === СПЕЦИАЛЬНЫЕ ТОКЕНЫ ===
    "<SIL>",    # тишина / пауза
]

# Специальный blank-токен для CTC (всегда последний)
BLANK_TOKEN = "<BLANK>"

# Полный словарь: фонемы + blank
ALL_TOKENS = RUSSIAN_PHONEMES + [BLANK_TOKEN]

# Маппинги индекс ↔ фонема
IDX2PHONE = {i: ph for i, ph in enumerate(ALL_TOKENS)}
PHONE2IDX = {ph: i for i, ph in IDX2PHONE.items()}

BLANK_IDX = PHONE2IDX[BLANK_TOKEN]
SIL_IDX   = PHONE2IDX["<SIL>"]

NUM_CLASSES = len(ALL_TOKENS)  # 46

# Кириллические буквы → список фонем (упрощённое G2P)
# Используется для генерации синтетических меток из текста
LETTER2PHONES = {
    "а": ["а"], "б": ["б"], "в": ["в"], "г": ["г"], "д": ["д"],
    "е": ["й", "э"], "ё": ["й", "о"], "ж": ["ж"], "з": ["з"],
    "и": ["и"], "й": ["й"], "к": ["к"], "л": ["л"], "м": ["м"],
    "н": ["н"], "о": ["о"], "п": ["п"], "р": ["р"], "с": ["с"],
    "т": ["т"], "у": ["у"], "ф": ["ф"], "х": ["х"], "ц": ["ц"],
    "ч": ["ч"], "ш": ["ш"], "щ": ["щ"], "ъ": [], "ы": ["ы"],
    "ь": [],    "э": ["э"], "ю": ["й", "у"], "я": ["й", "а"],
    " ": ["<SIL>"],
}

def text_to_phonemes(text: str) -> list[str]:
    """Очень упрощённое G2P: текст → список фонем (без ударения/ассимиляций)."""
    phones = []
    text = text.lower().strip()
    for ch in text:
        phones.extend(LETTER2PHONES.get(ch, []))
    return phones

def phonemes_to_indices(phones: list[str]) -> list[int]:
    return [PHONE2IDX[p] for p in phones if p in PHONE2IDX]

if __name__ == "__main__":
    print(f"Всего классов: {NUM_CLASSES}")
    print(f"Blank index: {BLANK_IDX}")
    example = "привет мир"
    ph = text_to_phonemes(example)
    idx = phonemes_to_indices(ph)
    print(f"'{example}' → {ph}")
    print(f"Индексы: {idx}")

Всего классов: 44
Blank index: 43
'привет мир' → ['п', 'р', 'и', 'в', 'й', 'э', 'т', '<SIL>', 'м', 'и', 'р']
Индексы: [6, 38, 2, 19, 41, 1, 8, 42, 33, 2, 38]


# Feature extraction

In [2]:
"""
Извлечение признаков из аудио для распознавания фонем.
Используем MFCC + дельты + дельта-дельты (40 × 3 = 120 признаков на фрейм).
"""

import numpy as np
import librosa
import torch
from dataclasses import dataclass


@dataclass
class AudioConfig:
    sample_rate: int = 16000       # Гц — стандарт для ASR
    n_mfcc: int = 40               # кол-во кепстральных коэффициентов
    n_fft: int = 512               # размер FFT-окна
    hop_length: int = 160          # шаг (10 мс при 16 кГц)
    win_length: int = 400          # длина окна (25 мс при 16 кГц)
    n_mels: int = 80               # мел-фильтров
    fmin: float = 80.0             # мин. частота (Гц)
    fmax: float = 7600.0           # макс. частота (Гц)
    use_deltas: bool = True        # добавлять дельты
    use_delta_deltas: bool = True  # добавлять дельта-дельты
    cmvn: bool = True              # Cepstral Mean-Variance Normalization


DEFAULT_CFG = AudioConfig()


def extract_mfcc(
    audio: np.ndarray,
    cfg: AudioConfig = DEFAULT_CFG,
) -> np.ndarray:
    """
    Извлекает MFCC-признаки из аудиосигнала.

    Args:
        audio: одномерный массив float32, амплитуды [-1, 1]
        cfg:   конфигурация

    Returns:
        features: np.ndarray формы (T, F), где
                  T — число фреймов,
                  F — число признаков (40, 80 или 120 при дельтах)
    """
    # 1. Нормализация амплитуды
    if audio.max() > 1.0 or audio.min() < -1.0:
        audio = audio / (np.abs(audio).max() + 1e-8)

    # 2. Предэмфаза — усиление высоких частот
    audio = np.append(audio[0], audio[1:] - 0.97 * audio[:-1])

    # 3. MFCC
    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=cfg.sample_rate,
        n_mfcc=cfg.n_mfcc,
        n_fft=cfg.n_fft,
        hop_length=cfg.hop_length,
        win_length=cfg.win_length,
        n_mels=cfg.n_mels,
        fmin=cfg.fmin,
        fmax=cfg.fmax,
    )  # (n_mfcc, T)

    features = [mfcc]

    # 4. Дельты (скорость изменения признаков)
    if cfg.use_deltas:
        delta1 = librosa.feature.delta(mfcc, order=1)
        features.append(delta1)

    # 5. Дельта-дельты (ускорение)
    if cfg.use_delta_deltas:
        delta2 = librosa.feature.delta(mfcc, order=2)
        features.append(delta2)

    # Объединяем по оси признаков: (F, T)
    features = np.concatenate(features, axis=0)

    # 6. CMVN — нормализация по среднему и дисперсии
    if cfg.cmvn:
        mean = features.mean(axis=1, keepdims=True)
        std  = features.std(axis=1, keepdims=True) + 1e-8
        features = (features - mean) / std

    # Транспонируем в (T, F) — удобнее для RNN
    return features.T.astype(np.float32)


def load_audio(path: str, cfg: AudioConfig = DEFAULT_CFG) -> np.ndarray:
    """Загружает аудиофайл и приводит к нужной частоте дискретизации."""
    audio, sr = librosa.load(path, sr=cfg.sample_rate, mono=True)
    return audio


def audio_to_features(
    path_or_array,
    cfg: AudioConfig = DEFAULT_CFG,
) -> torch.Tensor:
    """
    Полный пайплайн: путь к файлу ИЛИ numpy-массив → тензор признаков.

    Returns:
        Tensor формы (1, T, F) — готово для подачи в модель.
    """
    if isinstance(path_or_array, str):
        audio = load_audio(path_or_array, cfg)
    else:
        audio = path_or_array.astype(np.float32)

    feats = extract_mfcc(audio, cfg)          # (T, F)
    tensor = torch.from_numpy(feats)           # (T, F)
    return tensor.unsqueeze(0)                 # (1, T, F)


def get_feature_dim(cfg: AudioConfig = DEFAULT_CFG) -> int:
    """Возвращает размерность вектора признаков для одного фрейма."""
    dim = cfg.n_mfcc
    if cfg.use_deltas:
        dim += cfg.n_mfcc
    if cfg.use_delta_deltas:
        dim += cfg.n_mfcc
    return dim


# ──────────────────────────────────────────────
# Синтетическая генерация аудио для тестов
# ──────────────────────────────────────────────

def generate_sine_burst(
    freq: float = 440.0,
    duration: float = 0.2,
    sr: int = 16000,
    amplitude: float = 0.5,
) -> np.ndarray:
    """Генерирует синусоидальный сигнал — заглушка вместо реального аудио."""
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    wave = amplitude * np.sin(2 * np.pi * freq * t)
    # Плавное нарастание/спад (окно Ханна)
    envelope = np.hanning(len(wave))
    return (wave * envelope).astype(np.float32)


def generate_noise_segment(
    duration: float = 0.1,
    sr: int = 16000,
    amplitude: float = 0.05,
) -> np.ndarray:
    """Генерирует белый шум — заглушка для тишины."""
    n = int(sr * duration)
    return (amplitude * np.random.randn(n)).astype(np.float32)


if __name__ == "__main__":
    cfg = DEFAULT_CFG
    print(f"Размерность признаков: {get_feature_dim(cfg)}")

    # Тест на синтетическом сигнале
    audio = generate_sine_burst(freq=200, duration=0.5)
    feats = extract_mfcc(audio, cfg)
    print(f"Аудио: {len(audio)} сэмплов → признаки: {feats.shape}")
    # Ожидаем примерно (T≈50, F=120)

Размерность признаков: 120
Аудио: 8000 сэмплов → признаки: (51, 120)


# Model

In [4]:
"""
Архитектура нейросети для распознавания фонем русского языка.

Пайплайн:
    Аудио (16 кГц)
      ↓  extract_mfcc()
    Признаки (T × 120)
      ↓  ConvFrontend  — локальные акустические паттерны
    (T' × C)
      ↓  BiLSTM stack  — долгосрочные временны́е зависимости
    (T' × 2H)
      ↓  Linear + LogSoftmax
    Лог-вероятности (T' × NUM_CLASSES)
      ↓  CTC Loss / Greedy/Beam Decode
    Последовательность фонем

CTC (Connectionist Temporal Classification) не требует
выравнивания: модель учится сама определять длительность каждой фонемы.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional


# ──────────────────────────────────────────────────────────────────
# 1. Блок свёрточного фронт-энда
# ──────────────────────────────────────────────────────────────────

class ConvBlock(nn.Module):
    """Conv1d → BatchNorm → GELU → Dropout."""

    def __init__(self, in_ch: int, out_ch: int, kernel: int, stride: int = 1, dropout: float = 0.1):
        super().__init__()
        padding = kernel // 2
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel, stride=stride, padding=padding, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class ConvFrontend(nn.Module):
    """
    Трёхслойный свёрточный блок.
    Вход:  (B, T, F)
    Выход: (B, T', conv_channels[-1])

    Свёртки делают субдискретизацию по времени (stride=2 на первом слое),
    что снижает длину последовательности и ускоряет LSTM.
    """

    def __init__(
        self,
        input_dim: int,
        conv_channels: list[int] = [256, 256, 256],
        kernels: list[int] = [11, 7, 5],
        dropout: float = 0.1,
    ):
        super().__init__()
        channels = [input_dim] + conv_channels
        layers = []
        for i in range(len(conv_channels)):
            stride = 2 if i == 0 else 1   # субдискретизация на первом слое
            layers.append(ConvBlock(channels[i], channels[i+1], kernels[i], stride, dropout))
        self.layers = nn.Sequential(*layers)
        self.out_dim = conv_channels[-1]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, F) → Conv работает по каналам: нужно (B, F, T)
        x = x.transpose(1, 2)       # (B, F, T)
        x = self.layers(x)           # (B, C, T')
        x = x.transpose(1, 2)        # (B, T', C)
        return x


# ──────────────────────────────────────────────────────────────────
# 2. Основная модель
# ──────────────────────────────────────────────────────────────────

class RussianPhonemeNet(nn.Module):
    """
    Нейросеть распознавания фонем русского языка.

    Параметры:
        input_dim    — размерность MFCC-вектора (по умолчанию 120)
        conv_channels— список каналов свёрточных слоёв
        kernels      — размеры ядер свёрточных слоёв
        lstm_hidden  — размер скрытого состояния LSTM
        lstm_layers  — число слоёв LSTM
        dropout      — вероятность dropout
        num_classes  — число классов (фонем + blank)
    """

    def __init__(
        self,
        input_dim: int = 120,
        conv_channels: list[int] = [256, 256, 256],
        kernels: list[int] = [11, 7, 5],
        lstm_hidden: int = 512,
        lstm_layers: int = 4,
        dropout: float = 0.2,
        num_classes: int = NUM_CLASSES,
    ):
        super().__init__()
        self.input_dim = input_dim
        self.num_classes = num_classes

        # Свёрточный фронт-энд
        self.frontend = ConvFrontend(input_dim, conv_channels, kernels, dropout)

        # Двунаправленный LSTM
        self.lstm = nn.LSTM(
            input_size=self.frontend.out_dim,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )

        # Нормализация после LSTM
        self.layer_norm = nn.LayerNorm(lstm_hidden * 2)
        self.dropout = nn.Dropout(dropout)

        # Линейная проекция в пространство классов
        self.classifier = nn.Linear(lstm_hidden * 2, num_classes)

    def forward(
        self,
        x: torch.Tensor,
        lengths: Optional[torch.Tensor] = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            x:       (B, T, F) — батч MFCC-признаков
            lengths: (B,)      — реальные длины последовательностей (до паддинга)

        Returns:
            log_probs:      (T', B, C) — логарифмы вероятностей (для CTC)
            output_lengths: (B,)       — длины после субдискретизации
        """
        # Свёрточный фронт-энд
        x = self.frontend(x)          # (B, T', C)

        # Пересчитываем длины с учётом stride=2 первого Conv-блока
        if lengths is not None:
            output_lengths = ((lengths.float() + 1) / 2).long().clamp(min=1)
        else:
            output_lengths = torch.full(
                (x.size(0),), x.size(1), dtype=torch.long, device=x.device
            )

        # Packed sequence для эффективной обработки разных длин
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                x, output_lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            lstm_out, _ = self.lstm(packed)
            x, _ = nn.utils.rnn.pad_packed_sequence(lstm_out, batch_first=True)
        else:
            x, _ = self.lstm(x)       # (B, T', 2H)

        x = self.layer_norm(x)
        x = self.dropout(x)

        # Классификатор
        x = self.classifier(x)        # (B, T', num_classes)

        # CTC ожидает (T', B, C) и log-вероятности
        log_probs = F.log_softmax(x, dim=-1)
        log_probs = log_probs.transpose(0, 1)   # (T', B, C)

        return log_probs, output_lengths

    def count_params(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ──────────────────────────────────────────────────────────────────
# 3. CTC-декодер
# ──────────────────────────────────────────────────────────────────

def greedy_ctc_decode(
    log_probs: torch.Tensor,
    lengths: torch.Tensor,
    blank_idx: int = BLANK_IDX,
) -> list[list[int]]:
    """
    Жадный CTC-декодер: на каждом шаге берём argmax,
    затем схлопываем повторы и удаляем blank.

    Args:
        log_probs: (T, B, C) — логарифмы вероятностей
        lengths:   (B,)      — длины последовательностей

    Returns:
        Список списков индексов фонем для каждого примера в батче.
    """
    # Переводим в (B, T, C) для удобства
    probs = log_probs.transpose(0, 1)          # (B, T, C)
    batch_size = probs.size(0)
    results = []

    for b in range(batch_size):
        length = lengths[b].item()
        indices = probs[b, :length].argmax(dim=-1).tolist()   # (T,)

        # Схлопываем повторы и удаляем blank
        decoded = []
        prev = None
        for idx in indices:
            if idx != prev:
                if idx != blank_idx:
                    decoded.append(idx)
                prev = idx

        results.append(decoded)

    return results


def beam_search_ctc_decode(
    log_probs: torch.Tensor,
    lengths: torch.Tensor,
    blank_idx: int = BLANK_IDX,
    beam_size: int = 10,
) -> list[list[int]]:
    """
    Простой лучевой поиск (beam search) для CTC.
    Более точен, чем жадный декодер, особенно для коротких слов.

    Returns:
        Список лучших гипотез (индексы фонем) для каждого элемента батча.
    """
    probs = log_probs.transpose(0, 1).exp()   # (B, T, C), линейная шкала
    batch_size = probs.size(0)
    results = []

    for b in range(batch_size):
        T = lengths[b].item()
        # Состояние луча: (prefix_tuple, (prob_blank, prob_non_blank))
        beams = {(): (1.0, 0.0)}

        for t in range(T):
            p = probs[b, t]   # (C,)
            new_beams: dict = {}

            for prefix, (p_b, p_nb) in beams.items():
                p_total = p_b + p_nb

                # Добавляем blank — prefix не меняется
                nb = new_beams.setdefault(prefix, [0.0, 0.0])
                nb[0] += p_total * p[blank_idx].item()

                # Добавляем каждый не-blank символ
                for c in range(len(p)):
                    if c == blank_idx:
                        continue
                    c_prob = p[c].item()
                    new_prefix = prefix + (c,)

                    if len(prefix) > 0 and prefix[-1] == c:
                        # Тот же символ: только через blank
                        nb2 = new_beams.setdefault(new_prefix, [0.0, 0.0])
                        nb2[1] += p_b * c_prob
                    else:
                        nb2 = new_beams.setdefault(new_prefix, [0.0, 0.0])
                        nb2[1] += p_total * c_prob

            # Оставляем beam_size лучших
            beams = dict(
                sorted(
                    new_beams.items(),
                    key=lambda kv: kv[1][0] + kv[1][1],
                    reverse=True,
                )[:beam_size]
            )

        best_prefix = max(beams, key=lambda k: sum(beams[k]))
        results.append(list(best_prefix))

    return results


# ──────────────────────────────────────────────────────────────────
# 4. Быстрая проверка архитектуры
# ──────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Устройство: {device}")

    model = RussianPhonemeNet(input_dim=120).to(device)
    print(f"Параметров: {model.count_params():,}")

    # Синтетический батч: 4 примера, до 200 фреймов, 120 признаков
    B, T, F = 4, 200, 120
    x = torch.randn(B, T, F).to(device)
    lengths = torch.tensor([200, 180, 150, 120])

    log_probs, out_len = model(x, lengths)
    print(f"Вход:  {x.shape}")
    print(f"Выход: {log_probs.shape}  (T', B, C)")
    print(f"Длины: {out_len.tolist()}")

    # Greedy decode
    decoded = greedy_ctc_decode(log_probs, out_len)
    print(f"Декодировано (greedy, первый пример): {decoded[0][:10]}...")

    # CTC loss
    ctc = nn.CTCLoss(blank=BLANK_IDX, reduction="mean", zero_infinity=True)
    # Случайные целевые метки
    targets = torch.randint(0, NUM_CLASSES - 1, (B * 5,))
    target_lengths = torch.tensor([5, 5, 5, 5])
    loss = ctc(log_probs, targets, out_len, target_lengths)
    print(f"CTC loss (случайные метки): {loss.item():.4f}")
    print("✓ Архитектура в порядке")

Устройство: cpu
Параметров: 23,225,900


AttributeError: 'int' object has no attribute 'log_softmax'

# Dataset and dataloader

In [5]:
"""
Dataset и DataLoader для Sh1man/common_voice_21_ru (HuggingFace).

Структура датасета:
  - sample["mp3"]["array"]          — numpy float32, ресемплирован до 16 кГц
  - sample["mp3"]["sampling_rate"]  — 16000
  - sample["json"]["text"]          — русский текст транскрипции
  - sample["json"]["duration"]      — длительность в секундах
  - sample["__key__"]               — уникальный ID (common_voice_ru_XXXXXXXX)

Сплиты: train (93 531), validate (38 836), test (23 219)
"""

import re
import json as _json
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from features import AudioConfig, extract_mfcc, get_feature_dim
from features import generate_sine_burst, generate_noise_segment
from phonemes import text_to_phonemes, phonemes_to_indices


# ──────────────────────────────────────────────────────────────────
# Очистка текста
# ──────────────────────────────────────────────────────────────────

_KEEP = re.compile(r"[^а-яёА-ЯЁ ]")

def normalize_text(text: str) -> str:
    """
    Приводит текст к нижнему регистру, удаляет пунктуацию и
    лишние пробелы. «ё» сохраняется как отдельная буква.
    """
    text = text.lower()
    text = _KEEP.sub("", text)
    text = re.sub(r" +", " ", text)
    return text.strip()


# ──────────────────────────────────────────────────────────────────
# Аугментации
# ──────────────────────────────────────────────────────────────────

def augment_audio(audio: np.ndarray, sr: int = 16000) -> np.ndarray:
    """Лёгкие аугментации без pitch-shift (он медленный)."""
    # Случайная громкость (±6 дБ)
    if random.random() < 0.6:
        audio = audio * random.uniform(0.5, 2.0)
    # Аддитивный шум (SNR 20–40 дБ)
    if random.random() < 0.5:
        snr_db = random.uniform(20.0, 40.0)
        rms    = np.sqrt((audio ** 2).mean() + 1e-8)
        noise  = np.random.randn(len(audio)).astype(np.float32) * rms / (10 ** (snr_db / 20.0))
        audio  = audio + noise
    # Случайная обрезка краёв (до 5%)
    if random.random() < 0.3:
        m = max(1, int(0.05 * len(audio)))
        l, r = random.randint(0, m), random.randint(0, m)
        audio = audio[l: len(audio) - r or None]
    return np.clip(audio, -1.0, 1.0).astype(np.float32)


# ──────────────────────────────────────────────────────────────────
# Основной Dataset — Sh1man/common_voice_21_ru
# ──────────────────────────────────────────────────────────────────

class CommonVoice21Dataset(Dataset):
    """
    PyTorch Dataset поверх Sh1man/common_voice_21_ru.

    Args:
        split:        "train" | "validate" | "test"
        cfg:          конфигурация признаков (AudioConfig)
        max_duration: примеры длиннее пропускаются (сек)
        min_duration: примеры короче пропускаются (сек)
        augment:      включить аугментацию (рекомендуется только для train)
        max_samples:  взять первые N примеров (None = все; удобно для отладки)
        hf_cache_dir: папка кеша HuggingFace (None = ~/.cache/huggingface)
    """

    HF_DATASET_ID = "Sh1man/common_voice_21_ru"
    AUDIO_COL     = "mp3"
    TARGET_SR     = 16000

    def __init__(
        self,
        split: str = "train",
        cfg: AudioConfig = None,
        max_duration: float = 12.0,
        min_duration: float = 0.5,
        augment: bool = False,
        max_samples: int | None = None,
        hf_cache_dir: str | None = None,
    ):
        from datasets import load_dataset, Audio

        self.cfg        = cfg or AudioConfig()
        self.augment    = augment
        self.max_frames = int(max_duration * self.TARGET_SR)
        self.min_frames = int(min_duration * self.TARGET_SR)

        print(f"[CommonVoice21] Загружаем сплит '{split}' …")
        raw = load_dataset(
            self.HF_DATASET_ID,
            split=split,
            cache_dir=hf_cache_dir,
            trust_remote_code=True,
        )
        # Ресемплируем аудио к 16 кГц через HuggingFace
        raw = raw.cast_column(self.AUDIO_COL, Audio(sampling_rate=self.TARGET_SR))

        if max_samples is not None:
            raw = raw.select(range(min(max_samples, len(raw))))

        self._hf = raw
        print(f"[CommonVoice21] Готово: {len(self._hf):,} примеров.")

    def __len__(self) -> int:
        return len(self._hf)

    def __getitem__(self, idx: int) -> dict | None:
        row = self._hf[idx]

        # ── Аудио ────────────────────────────────────────────────
        audio: np.ndarray = row[self.AUDIO_COL]["array"].astype(np.float32)

        # Фильтр по длине
        if len(audio) < self.min_frames or len(audio) > self.max_frames:
            return None

        # Нормализация пика
        peak = np.abs(audio).max()
        if peak > 1e-6:
            audio = audio / peak

        if self.augment:
            audio = augment_audio(audio, self.TARGET_SR)

        # ── Текст → фонемы ───────────────────────────────────────
        json_meta = row["json"]
        if isinstance(json_meta, str):
            json_meta = _json.loads(json_meta)

        raw_text  = json_meta.get("text", "")
        text      = normalize_text(raw_text)
        if not text:
            return None

        phone_ids = phonemes_to_indices(text_to_phonemes(text))
        if len(phone_ids) == 0:
            return None

        # ── MFCC-признаки ────────────────────────────────────────
        features = extract_mfcc(audio, self.cfg)   # (T, F)

        # CTC требует T >= len(targets)
        if features.shape[0] < len(phone_ids):
            return None

        return {
            "features":   torch.from_numpy(features),
            "targets":    torch.tensor(phone_ids, dtype=torch.long),
            "feat_len":   features.shape[0],
            "target_len": len(phone_ids),
            "text":       raw_text,
            "id":         row.get("__key__", str(idx)),
        }


# ──────────────────────────────────────────────────────────────────
# Синтетический Dataset (для unit-тестов без интернета)
# ──────────────────────────────────────────────────────────────────

PHONEME_FREQS = {
    "а": 800, "э": 500, "и": 300, "о": 600, "у": 400, "ы": 450,
    "п": 100, "б": 120, "т": 150, "д": 130, "к": 200, "г": 180,
    "с": 4000, "з": 3500, "ш": 3000, "ж": 2800, "х": 5000,
    "м": 250,  "н": 270,  "л": 340,  "р": 380,  "й": 320,
    "ц": 3800, "ч": 3600, "щ": 3200, "в": 1200, "ф": 1500,
}

class SyntheticPhonemeDataset(Dataset):
    """Синтетический датасет из синусоид — для быстрой проверки кода."""

    def __init__(self, size: int = 500, cfg: AudioConfig = None):
        self.size     = size
        self.cfg      = cfg or AudioConfig()
        self.phonemes = list(PHONEME_FREQS.keys())

    def __len__(self) -> int:
        return self.size

    def __getitem__(self, idx: int) -> dict:
        n      = random.randint(3, 10)
        chosen = [random.choice(self.phonemes) for _ in range(n)]

        segs = [generate_noise_segment(0.05, self.cfg.sample_rate, 0.02)]
        for ph in chosen:
            segs.append(generate_sine_burst(PHONEME_FREQS[ph], random.uniform(0.08, 0.18), self.cfg.sample_rate))
            segs.append(generate_noise_segment(0.03, self.cfg.sample_rate, 0.01))
        audio = np.concatenate(segs)
        audio += 0.01 * np.random.randn(len(audio)).astype(np.float32)

        features  = extract_mfcc(audio, self.cfg)
        phone_ids = phonemes_to_indices(chosen)
        return {
            "features":   torch.from_numpy(features),
            "targets":    torch.tensor(phone_ids, dtype=torch.long),
            "feat_len":   features.shape[0],
            "target_len": len(phone_ids),
            "text":       " ".join(chosen),
            "id":         f"synth_{idx}",
        }


# ──────────────────────────────────────────────────────────────────
# Collate function и DataLoader
# ──────────────────────────────────────────────────────────────────

def collate_fn(batch: list) -> dict | None:
    """Склеивает примеры в батч с паддингом; None-элементы отбрасываются."""
    batch = [b for b in batch if b is not None]
    if not batch:
        return None
    # Сортировка по убыванию длины (нужна для pack_padded_sequence)
    batch.sort(key=lambda x: x["feat_len"], reverse=True)

    return {
        "features":    pad_sequence([b["features"] for b in batch], batch_first=True),
        "feat_lens":   torch.tensor([b["feat_len"]   for b in batch], dtype=torch.long),
        "targets":     torch.cat([b["targets"] for b in batch]),
        "target_lens": torch.tensor([b["target_len"] for b in batch], dtype=torch.long),
        "texts":       [b["text"] for b in batch],
        "ids":         [b["id"]   for b in batch],
    }


def make_dataloader(
    dataset: Dataset,
    batch_size: int = 16,
    shuffle: bool = True,
    num_workers: int = 0,
) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=collate_fn,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(num_workers > 0),
    )


# ──────────────────────────────────────────────────────────────────
# Быстрая проверка
# ──────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    cfg = AudioConfig()
    print(f"Размерность признаков: {get_feature_dim(cfg)}")

    ds     = SyntheticPhonemeDataset(size=32, cfg=cfg)
    loader = make_dataloader(ds, batch_size=8, shuffle=False)
    batch  = next(iter(loader))
    print(f"Синтетический батч: features={batch['features'].shape}, "
          f"feat_lens={batch['feat_lens'].tolist()}")

    # Раскомментировать для проверки реального датасета:
    # ds_real = CommonVoice21Dataset(split="train", max_samples=50)
    # s = ds_real[0]
    # print(f"features={s['features'].shape}, text='{s['text']}'")

ModuleNotFoundError: No module named 'features'

# Training

In [ ]:
"""
Обучение модели распознавания фонем на датасете Sh1man/common_voice_21_ru.

Быстрый старт
─────────────
# Проверка кода (синтетические данные, без скачивания):
python train.py --synthetic --epochs 5 --batch_size 32

# Полное обучение (автоматически скачает датасет с HuggingFace):
python train.py --epochs 100 --batch_size 16 --lr 3e-4

# Только первые 5000 примеров (удобно для первых экспериментов):
python train.py --max_samples 5000 --epochs 30

# Продолжить с чекпоинта:
python train.py --resume checkpoints/best_model.pt
"""

import os
import time
import json
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR

from features import AudioConfig, get_feature_dim
from dataset import CommonVoice21Dataset, SyntheticPhonemeDataset, make_dataloader
from model import RussianPhonemeNet, greedy_ctc_decode
from phonemes import BLANK_IDX, IDX2PHONE, NUM_CLASSES


# ──────────────────────────────────────────────────────────────────
# Phone Error Rate (PER)
# ──────────────────────────────────────────────────────────────────

def _levenshtein(a: list, b: list) -> int:
    m, n = len(a), len(b)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev, dp[0] = dp[:], i
        for j in range(1, n + 1):
            dp[j] = prev[j-1] if a[i-1] == b[j-1] else 1 + min(prev[j], dp[j-1], prev[j-1])
    return dp[n]


def phone_error_rate(
    decoded_batch: list[list[int]],
    targets: torch.Tensor,
    target_lengths: torch.Tensor,
) -> float:
    total_edit = total_len = 0
    offset = 0
    for hyp, t_len in zip(decoded_batch, target_lengths.tolist()):
        ref = targets[offset: offset + t_len].tolist()
        offset += t_len
        total_edit += _levenshtein(hyp, ref)
        total_len  += max(len(ref), 1)
    return total_edit / max(total_len, 1)


# ──────────────────────────────────────────────────────────────────
# Одна эпоха (обучение или валидация)
# ──────────────────────────────────────────────────────────────────

def run_epoch(
    model: nn.Module,
    loader,
    ctc_loss: nn.CTCLoss,
    optimizer: optim.Optimizer = None,
    scheduler=None,
    device: str = "cpu",
    train: bool = True,
    grad_clip: float = 5.0,
    log_every: int = 200,
) -> dict:
    model.train(train)
    total_loss = total_per = n_batches = 0

    with torch.set_grad_enabled(train):
        for step, batch in enumerate(loader, 1):
            if batch is None:
                continue

            features    = batch["features"].to(device)
            feat_lens   = batch["feat_lens"].to(device)
            targets     = batch["targets"].to(device)
            target_lens = batch["target_lens"].to(device)

            log_probs, out_lens = model(features, feat_lens)
            loss = ctc_loss(log_probs, targets, out_lens, target_lens)

            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
                if scheduler:
                    scheduler.step()

            with torch.no_grad():
                decoded = greedy_ctc_decode(log_probs.detach(), out_lens)
                per = phone_error_rate(decoded, targets.cpu(), target_lens.cpu())

            total_loss += loss.item()
            total_per  += per
            n_batches  += 1

            if train and step % log_every == 0:
                avg_loss = total_loss / n_batches
                avg_per  = total_per  / n_batches
                lr_now   = scheduler.get_last_lr()[0] if scheduler else 0
                print(f"    шаг {step:5d} | loss {avg_loss:.4f} | PER {avg_per:.4f} | lr {lr_now:.2e}")

    n = max(n_batches, 1)
    return {"loss": total_loss / n, "per": total_per / n}


# ──────────────────────────────────────────────────────────────────
# Вывод примеров распознавания (для мониторинга качества)
# ──────────────────────────────────────────────────────────────────

def show_examples(model, loader, device, n=3):
    model.eval()
    shown = 0
    with torch.no_grad():
        for batch in loader:
            if batch is None or shown >= n:
                break
            features  = batch["features"].to(device)
            feat_lens = batch["feat_lens"].to(device)
            log_probs, out_lens = model(features, feat_lens)
            decoded = greedy_ctc_decode(log_probs, out_lens)
            for i in range(min(n - shown, len(decoded))):
                phones_str = " ".join(IDX2PHONE.get(idx, "?") for idx in decoded[i])
                print(f"    Текст:   {batch['texts'][i]}")
                print(f"    Фонемы:  {phones_str}")
                print()
                shown += 1


# ──────────────────────────────────────────────────────────────────
# Главный цикл обучения
# ──────────────────────────────────────────────────────────────────

def train(args):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Устройство: {device}")

    cfg       = AudioConfig()
    input_dim = get_feature_dim(cfg)
    print(f"Размерность MFCC-признаков: {input_dim}")

    # ── Датасеты ──────────────────────────────────────────────────
    if args.synthetic:
        print("Режим: синтетические данные")
        train_ds = SyntheticPhonemeDataset(size=2000, cfg=cfg)
        val_ds   = SyntheticPhonemeDataset(size=400,  cfg=cfg)
    else:
        print(f"Датасет: {CommonVoice21Dataset.HF_DATASET_ID}")
        train_ds = CommonVoice21Dataset(
            split="train",
            cfg=cfg,
            augment=True,
            max_samples=args.max_samples,
            hf_cache_dir=args.hf_cache_dir,
        )
        val_ds = CommonVoice21Dataset(
            split="validate",
            cfg=cfg,
            augment=False,
            # Валидационную выборку тоже можно ограничить для скорости
            max_samples=args.max_val_samples,
            hf_cache_dir=args.hf_cache_dir,
        )

    train_loader = make_dataloader(train_ds, args.batch_size, shuffle=True,  num_workers=args.workers)
    val_loader   = make_dataloader(val_ds,   args.batch_size, shuffle=False, num_workers=args.workers)
    print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,} примеров")

    # ── Модель ────────────────────────────────────────────────────
    model = RussianPhonemeNet(
        input_dim=input_dim,
        conv_channels=[256, 256, 256],
        kernels=[11, 7, 5],
        lstm_hidden=args.lstm_hidden,
        lstm_layers=args.lstm_layers,
        dropout=args.dropout,
        num_classes=NUM_CLASSES,
    ).to(device)
    print(f"Параметров: {model.count_params():,}")

    # ── Чекпоинт ──────────────────────────────────────────────────
    start_epoch = 0
    best_per    = float("inf")
    if args.resume and os.path.exists(args.resume):
        ckpt = torch.load(args.resume, map_location=device)
        model.load_state_dict(ckpt["model"])
        start_epoch = ckpt.get("epoch", 0) + 1
        best_per    = ckpt.get("best_per", float("inf"))
        print(f"Восстановлено с эпохи {start_epoch}, best PER={best_per:.4f}")

    # ── Оптимизатор ───────────────────────────────────────────────
    optimizer = optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    total_steps = (args.epochs - start_epoch) * len(train_loader)
    scheduler = OneCycleLR(
        optimizer,
        max_lr=args.lr,
        total_steps=max(total_steps, 1),
        pct_start=0.05,
        anneal_strategy="cos",
    )
    ctc_loss = nn.CTCLoss(blank=BLANK_IDX, reduction="mean", zero_infinity=True)

    # ── История ───────────────────────────────────────────────────
    history: list[dict] = []
    os.makedirs(args.save_dir, exist_ok=True)

    # ── Цикл эпох ─────────────────────────────────────────────────
    print(f"\nОбучение: {args.epochs - start_epoch} эпох(и)\n" + "─" * 65)

    for epoch in range(start_epoch, args.epochs):
        t0 = time.time()

        print(f"Эпоха {epoch+1}/{args.epochs} — обучение …")
        tr = run_epoch(
            model, train_loader, ctc_loss, optimizer, scheduler,
            device, train=True, grad_clip=args.grad_clip, log_every=args.log_every
        )

        print(f"Эпоха {epoch+1}/{args.epochs} — валидация …")
        va = run_epoch(model, val_loader, ctc_loss, device=device, train=False)

        elapsed = time.time() - t0
        lr_now  = scheduler.get_last_lr()[0]

        row = {
            "epoch":      epoch + 1,
            "train_loss": round(tr["loss"], 4),
            "train_per":  round(tr["per"],  4),
            "val_loss":   round(va["loss"],  4),
            "val_per":    round(va["per"],   4),
            "lr":         round(lr_now, 8),
            "time_s":     round(elapsed, 1),
        }
        history.append(row)

        print(
            f"  ✦ loss {tr['loss']:.4f}/{va['loss']:.4f} | "
            f"PER {tr['per']:.4f}/{va['per']:.4f} | "
            f"lr {lr_now:.2e} | {elapsed:.1f}с"
        )

        # Несколько примеров распознавания
        if (epoch + 1) % args.show_examples_every == 0:
            print("  — Примеры распознавания (val):")
            show_examples(model, val_loader, device, n=2)

        # Сохраняем лучшую модель
        if va["per"] < best_per:
            best_per = va["per"]
            path = os.path.join(args.save_dir, "best_model.pt")
            torch.save({
                "epoch":    epoch,
                "model":    model.state_dict(),
                "best_per": best_per,
                "config":   cfg.__dict__,
                "args":     vars(args),
            }, path)
            print(f"  ✓ best_model.pt сохранён (PER={best_per:.4f})")

        # Регулярные чекпоинты
        if (epoch + 1) % args.save_every == 0:
            p = os.path.join(args.save_dir, f"checkpoint_ep{epoch+1}.pt")
            torch.save({"epoch": epoch, "model": model.state_dict()}, p)

        # История
        with open(os.path.join(args.save_dir, "history.json"), "w", encoding="utf-8") as f:
            json.dump(history, f, ensure_ascii=False, indent=2)

    print("\n" + "─" * 65)
    print(f"Обучение завершено. Лучший val PER: {best_per:.4f}")
    print(f"Чекпоинты → {args.save_dir}/")


# ──────────────────────────────────────────────────────────────────
# Инференс одного аудиофайла
# ──────────────────────────────────────────────────────────────────

def inference(args):
    """Распознать фонемы в одном аудиофайле."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    cfg       = AudioConfig()
    input_dim = get_feature_dim(cfg)

    model = RussianPhonemeNet(input_dim=input_dim, num_classes=NUM_CLASSES).to(device)
    ckpt  = torch.load(args.checkpoint, map_location=device)
    model.load_state_dict(ckpt["model"])
    model.eval()
    print(f"Модель загружена из {args.checkpoint}")

    from features import audio_to_features
    features = audio_to_features(args.audio_file, cfg).to(device)  # (1, T, F)

    with torch.no_grad():
        log_probs, out_lens = model(features)
        decoded = greedy_ctc_decode(log_probs, out_lens)

    phones = [IDX2PHONE.get(i, "?") for i in decoded[0]]
    print(f"Файл:   {args.audio_file}")
    print(f"Фонемы: {' '.join(phones)}")


# ──────────────────────────────────────────────────────────────────
# Аргументы CLI
# ──────────────────────────────────────────────────────────────────

def parse_args():
    p = argparse.ArgumentParser(
        description="Обучение / инференс модели фонем (common_voice_21_ru)"
    )
    sub = p.add_subparsers(dest="command")

    # ─ train ─────────────────────────────────────────────────────
    t = sub.add_parser("train", help="Обучение модели")
    t.add_argument("--synthetic",          action="store_true",
                   help="Синтетический датасет (без интернета)")
    t.add_argument("--max_samples",        type=int,   default=None,
                   help="Кол-во примеров train (None=все)")
    t.add_argument("--max_val_samples",    type=int,   default=None,
                   help="Кол-во примеров val  (None=все)")
    t.add_argument("--hf_cache_dir",       type=str,   default=None,
                   help="Папка кеша HuggingFace")
    t.add_argument("--workers",            type=int,   default=0)
    t.add_argument("--lstm_hidden",        type=int,   default=512)
    t.add_argument("--lstm_layers",        type=int,   default=4)
    t.add_argument("--dropout",            type=float, default=0.2)
    t.add_argument("--epochs",             type=int,   default=100)
    t.add_argument("--batch_size",         type=int,   default=16)
    t.add_argument("--lr",                 type=float, default=3e-4)
    t.add_argument("--grad_clip",          type=float, default=5.0)
    t.add_argument("--save_dir",           type=str,   default="checkpoints")
    t.add_argument("--save_every",         type=int,   default=10)
    t.add_argument("--log_every",          type=int,   default=200)
    t.add_argument("--show_examples_every",type=int,   default=5)
    t.add_argument("--resume",             type=str,   default=None)

    # ─ infer ─────────────────────────────────────────────────────
    i = sub.add_parser("infer", help="Распознать фонемы в аудиофайле")
    i.add_argument("audio_file",  type=str, help="Путь к WAV/MP3")
    i.add_argument("--checkpoint", type=str, default="checkpoints/best_model.pt")

    return p.parse_args()


# ──────────────────────────────────────────────────────────────────
# Точка входа
# ──────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    args = parse_args()

    if args.command == "train" or args.command is None:
        # Если команда не указана — обучение с параметрами по умолчанию
        if args.command is None:
            import sys
            # Перезапуск с подкомандой train для backward-совместимости
            sys.argv.insert(1, "train")
            args = parse_args()
        train(args)
    elif args.command == "infer":
        inference(args)
    else:
        import argparse
        argparse.ArgumentParser().print_help()